# Multi-Agent News Brief

Find news, filter it with an editor, write a script, and generate a voice briefing with OpenAI or Gemini.

## Workflow overview

This workflow searches for recent news, uses two agents to refine it into a script, then creates an audio briefing. Run the next cell to render the diagram.

In [18]:
%%html
<script src="https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.min.js"></script>

<div class="mermaid">
flowchart TD
    A[Choose a news topic] --> B[DuckDuckGo news search]
    B --> C[News Editor agent: filter and refine]
    C --> D[News Script Writer agent]
    D --> E{Selected provider}
    E -->|OpenAI| F[OpenAI TTS: MP3]
    E -->|Gemini| G[Gemini TTS: WAV]
    F --> H[Play and download audio]
    G --> H
</div>

<script>
  mermaid.initialize({ startOnLoad: false, theme: 'neutral' });
  mermaid.run({ querySelector: '.mermaid' });
</script>

## 1. Install dependencies

In [19]:
%pip install -q openai-agents ddgs google-genai

## 2. Select an agent provider

Set `PROVIDER` to `"openai"` or `"gemini"`. The same provider also generates the final voice briefing.

In [23]:
import os

from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from google.colab import userdata
from openai import AsyncOpenAI

PROVIDER = "gemini"  # Change to "gemini" to use Gemini for agents and TTS.
OPENAI_MODEL_NAME = "gpt-5-mini"
GEMINI_MODEL_NAME = "gemini-3.8-flash"

if PROVIDER == "openai":
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    if not OPENAI_API_KEY:
        raise ValueError("Add OPENAI_API_KEY to Colab Secrets.")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    model = OPENAI_MODEL_NAME
elif PROVIDER == "gemini":
    GEMINI_API_KEY = userdata.get("GeminiAPIKeysl")
    if not GEMINI_API_KEY:
        raise ValueError("Add GEMINI_API_KEY to Colab Secrets for Gemini agents.")
    set_tracing_disabled(disabled=True)
    model = OpenAIChatCompletionsModel(
        model=GEMINI_MODEL_NAME,
        openai_client=AsyncOpenAI(
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            api_key=GEMINI_API_KEY,
        ),
    )
else:
    raise ValueError("PROVIDER must be 'openai' or 'gemini'.")

## 3. Find recent news

In [30]:
from ddgs import DDGS


def search_news(query: str) -> str:
    """Find recent news and return each item's title, summary, date, and URL."""
    results = DDGS(timeout=15).news(query, timelimit="d", max_results=10)
    if not results:
        raise RuntimeError("No recent news results were found. Try a different topic.")

    for index, item in enumerate(results, start=1):
        print(f"{index}. {item.get('title', 'Untitled')}")
        print(item.get('url', 'No URL available.'))

    return "\n\n".join(
        f"Title: {item.get('title', 'Untitled')}\nDate: {item.get('date', 'Unknown')}\n"
        f"Summary: {item.get('body', 'No summary available.')}\nURL: {item.get('url', '')}"
        for item in results
    )


topic = "Traveling Destinations"
news_items = search_news(topic)

1. The golden rules of cruising: 10 things to know before you sail
https://www.msn.com/en-gb/news/other/the-golden-rules-of-cruising-10-things-to-know-before-you-sail/ar-AA2c7gnD?ocid=BingNewsVerp
2. Separate but inter-dependent: The four quarters of Nigeria’s campaign season
https://www.thecable.ng/separate-but-inter-dependent-the-four-quarters-of-nigerias-campaign-season/
3. Two spectacular Welsh road trips named among UK’s most popular
https://nation.cymru/news/two-spectacular-welsh-road-trips-named-among-uks-most-popular/


## 4. Editor Agent: filter and refine

In [32]:
from agents import Agent, Runner

editor_agent = Agent(
    name="News Editor",
    instructions="Select the three most relevant, credible, non-duplicative items. Exclude ads, clickbait, speculation, and weak evidence. Preserve each selected item's key facts, date, and URL.",
    model=model,
)

editor_result = await Runner.run(editor_agent, f"Topic: {topic}\n\nNews items:\n{news_items}")
edited_news = editor_result.final_output
print(edited_news)

Here are the most relevant, credible, and non-duplicative items related to travel destinations from the provided list:

* **Two spectacular Welsh road trips named among UK’s most popular**  
  **Date:** 2026-07-26T08:10:20+00:00  
  **Summary:** Two scenic Welsh road trip routes have been ranked among the most popular driving holiday destinations in the UK and Ireland.  
  **URL:** https://nation.cymru/news/two-spectacular-welsh-road-trips-named-among-uks-most-popular/

* **The golden rules of cruising: 10 things to know before you sail**  
  **Date:** 2026-07-27T08:10:20+00:00  
  **Summary:** A cruise specialist draws from over 100 trips to share essential advice, tips, and guidelines for travelers planning a cruise vacation.  
  **URL:** https://www.msn.com/en-gb/news/other/the-golden-rules-of-cruising-10-things-to-know-before-you-sail/ar-AA2c7gnD?ocid=BingNewsVerp

*(Note: The remaining article regarding Nigeria's political campaign season was excluded due to irrelevance to the top

## 5. Writer Agent: create the script

In [36]:
writer_agent = Agent(
    name="News Script Writer",
    instructions="Write a neutral 45- to 60-second spoken news script using only the editor's selected items. Do not add unsupported facts. End by naming the source publications without reading URLs aloud in sinhala language.",
    model=model,
)

writer_result = await Runner.run(writer_agent, f"Create a spoken news script from this edited brief:\n\n{edited_news}")
news_script = writer_result.final_output
print(news_script)

ආයුබෝවන්, සංචාරක පුවත් සංක්ෂිප්තය වෙත ඔබව සාදරයෙන් පිළිගනිමු. 

එක්සත් රාජධානිය සහ අයර්ලන්තය තුළ වඩාත්ම ජනප්‍රිය රිය පැදවීමේ නිවාඩු ගමනාන්ත අතරට වේල්සයේ මනරම් මාර්ග සංචාරක මාර්ග දෙකක් නම් කර තිබෙනවා. 

මේ අතර, නෞකා සංචාරක නිවාඩුවක් සැලසුම් කරන සංචාරකයින් වෙනුවෙන් නෞකා විශේෂඥයෙකු විසින් වැදගත් උපදෙස් සහ මඟපෙන්වීම් ඉදිරිපත් කර තිබෙනවා. ගමන් වාර සියයකට අධික අත්දැකීම් ඇසුරින්, නෞකාවකින් පිටත්ව යාමට පෙර සංචාරකයින් දැනගත යුතු වැදගත් කරුණු දහයක් එම විශේෂඥයා විසින් පෙන්වා දී තිබේ. 

මෙම තොරතුරු වාර්තා කර තිබෙන්නේ Nation.Cymru සහ MSN යන ප්‍රකාශන මගිනි.


## 6. Generate the voice briefing

This creates AI-generated speech with the selected provider. Disclose that the voice is AI-generated when sharing it.

In [39]:
from IPython.display import Audio, display

if PROVIDER == "openai":
    tts_client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    speech = await tts_client.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="coral",
        input=news_script,
        instructions="Speak clearly in a calm, neutral broadcast-news style in sinhala language.",
    )
    audio_path = "news_brief.mp3"
    speech.write_to_file(audio_path)
else:
    import base64
    import wave

    from google import genai

    def save_wav(filename, pcm, channels=1, rate=24000, sample_width=2):
        with wave.open(filename, "wb") as wav_file:
            wav_file.setnchannels(channels)
            wav_file.setsampwidth(sample_width)
            wav_file.setframerate(rate)
            wav_file.writeframes(pcm)

    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
    response = gemini_client.interactions.create(
        model="gemini-3.1-flash-tts-preview",
        input=(
            "Read this in a calm, neutral broadcast-news style:\n\n"
            f"{news_script}"
        ),
        response_format={"type": "audio"},
        generation_config={"speech_config": [{"voice": "Kore"}]},
    )
    audio_path = "news_brief.wav"
    save_wav(audio_path, base64.b64decode(response.output_audio.data))

display(Audio(audio_path))


## 7. Download the audio file

Run this cell to save the generated MP3 (OpenAI) or WAV (Gemini) file to your computer.

In [ ]:
from google.colab import files

files.download(audio_path)